# Klasifikasi Vokal SIBI Berbasis Landmark MediaPipe

Notebook ini merupakan perbaikan pipeline klasifikasi lima vokal SIBI (A, E, I, O, U) dengan deduplikasi berbasis hash, pengelompokan foto beruntun, pemisahan train/validation/test berbasis group, evaluasi coverage deteksi, serta penyimpanan model untuk inference.

## 1. Import Library

In [ ]:
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import urllib.request
from collections import Counter
from datetime import datetime

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from PIL import ExifTags, Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers

print("Python      :", platform.python_version())
print("OS          :", platform.platform())
print("Processor   :", platform.processor())
print("TensorFlow  :", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices("GPU"))

## 2. Konfigurasi

In [ ]:
BASE_DIR = "."
DATASET_DIR = os.path.join(BASE_DIR, "dataset")
OLD_EXP_DIR = os.path.join(BASE_DIR, "hasil_eksperimen")
OUTPUT_DIR = os.path.join(BASE_DIR, "output_final")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASSES = ["A", "E", "I", "O", "U"]
EXTS = (".jpg", ".jpeg", ".png", ".webp")
SEED = 42

# Ambang pengelompokan foto beruntun (timestamp + kemiripan visual)
GROUP_MAX_GAP_SEC = 1.0
GROUP_MIN_CORR = 0.85
GROUP_MAX_MAE = 20.0
GROUP_MIN_ANCHOR_CORR = 0.75

# Konfigurasi MediaPipe Hand Landmarker
HAND_TASK_FILENAME = "hand_landmarker.task"
HAND_TASK_URL = (
    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
    "hand_landmarker/float16/1/hand_landmarker.task"
)
MP_NUM_HANDS = 1
MP_MIN_DET_CONF = 0.3
MP_MIN_PRES_CONF = 0.3
MP_MIN_TRACK_CONF = 0.3

# Parameter augmentasi (hanya pada data train)
N_AUG_PER_SAMPLE = 2
AUG_SCALE_MIN = 0.92
AUG_SCALE_MAX = 1.08
AUG_NOISE_STD = 0.015

# Hyperparameter training E0 dan E1
EPOCHS = 80
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EARLY_STOP_PATIENCE = 12


def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


set_all_seeds(SEED)
sns.set_style("whitegrid")

## 3. Pemeriksaan Data, Duplikasi, dan Pengelompokan

In [ ]:
def hitung_sha256(path_file):
    h = hashlib.sha256()
    with open(path_file, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def parse_timestamp_nama(nama_file):
    m = re.match(r"^(\d{8}_\d{6})(?:\((\d+)\))?\.[^.]+$", nama_file, re.IGNORECASE)
    if not m:
        return None, -1
    dt = datetime.strptime(m.group(1), "%Y%m%d_%H%M%S")
    sub_idx = int(m.group(2)) if m.group(2) is not None else -1
    return dt, sub_idx


raw_records = []
for cls in CLASSES:
    cls_dir = os.path.join(DATASET_DIR, cls)
    files = sorted(
        f for f in os.listdir(cls_dir)
        if os.path.isfile(os.path.join(cls_dir, f)) and f.lower().endswith(EXTS)
    )
    for fname in files:
        fpath = os.path.join(cls_dir, fname)
        rel_path = os.path.relpath(fpath, BASE_DIR).replace("\\", "/")
        img = cv2.imread(fpath)
        valid_img = img is not None
        fhash = hitung_sha256(fpath)
        dt, sub_idx = parse_timestamp_nama(fname)
        raw_records.append({
            "path_relatif": rel_path,
            "abs_path": fpath,
            "nama_file": fname,
            "label": cls,
            "valid_img": valid_img,
            "hash": fhash,
            "dt": dt,
            "ts": dt.timestamp() if dt is not None else np.nan,
            "sub_idx": sub_idx,
        })

df_raw = pd.DataFrame(raw_records)
print(f"Total file terbaca : {len(df_raw)}")
print(f"Citra valid        : {int(df_raw['valid_img'].sum())}")
print(f"Hash unik          : {df_raw['hash'].nunique()}")
print("Distribusi awal    :", df_raw["label"].value_counts().reindex(CLASSES).to_dict())

In [ ]:
# Pemeriksaan metadata EXIF dan dimensi citra
dim_counter = Counter()
camera_counter = Counter()

for p in df_raw["abs_path"]:
    with Image.open(p) as im:
        dim_counter[im.size] += 1
        exif = im.getexif()
        make = exif.get(271, "")
        model = exif.get(272, "")
        camera_counter[f"{make} {model}".strip()] += 1

print("Dimensi citra:", dict(dim_counter))
print("Perangkat EXIF:", dict(camera_counter))
print("Informasi subjek/sesi eksplisit: tidak tersedia di nama file maupun EXIF.")

In [ ]:
# Penanganan duplikat berbasis SHA-256
hash_groups = df_raw.groupby("hash")
alasan_keluar = {}
dup_diff_rows = []
dup_same_count = 0

for h_val, grp in hash_groups:
    if len(grp) == 1:
        continue
    labels_unik = grp["label"].unique()
    if len(labels_unik) == 1:
        # Simpan satu salinan pertama, keluarkan sisanya
        dup_same_count += len(grp) - 1
        for idx in grp.index[1:]:
            alasan_keluar[idx] = "duplikat_identik_label_sama"
    else:
        # Keluarkan semua salinan jika label konflik
        for idx in grp.index:
            alasan_keluar[idx] = "duplikat_identik_label_berbeda"
            dup_diff_rows.append({
                "hash": h_val[:12],
                "label": df_raw.loc[idx, "label"],
                "path_relatif": df_raw.loc[idx, "path_relatif"],
            })

df_raw["alasan_dikeluarkan"] = df_raw.index.map(lambda i: alasan_keluar.get(i, ""))
df_clean = df_raw[df_raw["valid_img"] & (df_raw["alasan_dikeluarkan"] == "")].copy()

print(f"Duplikat identik label sama dikeluarkan    : {dup_same_count}")
print(f"Duplikat identik label berbeda dikeluarkan : {len(dup_diff_rows)} file ({len(dup_diff_rows)//2} pasangan)")
print(f"Total data bersih siap diproses            : {len(df_clean)}")
if dup_diff_rows:
    display(pd.DataFrame(dup_diff_rows))

In [ ]:
# Pembentukan group untuk mencegah kebocoran foto beruntun antar-split
def muat_thumbnail_gray(path_file):
    img = cv2.imread(path_file, cv2.IMREAD_REDUCED_GRAYSCALE_8)
    if img is None:
        img = cv2.imread(path_file, cv2.IMREAD_GRAYSCALE)
    small = cv2.resize(img, (64, 48), interpolation=cv2.INTER_AREA).astype(np.float32)
    return small.ravel()


def korelasi_dan_mae(v1, v2):
    mae = float(np.mean(np.abs(v1 - v2)))
    c1 = v1 - v1.mean()
    c2 = v2 - v2.mean()
    denom = float(np.linalg.norm(c1) * np.linalg.norm(c2)) + 1e-8
    corr = float(np.dot(c1, c2) / denom)
    return corr, mae


df_clean = df_clean.sort_values(["label", "ts", "sub_idx", "nama_file"]).reset_index(drop=True)
thumbs = [muat_thumbnail_gray(p) for p in df_clean["abs_path"]]

group_ids = []
contoh_grup = []

for cls in CLASSES:
    idxs = df_clean.index[df_clean["label"] == cls].tolist()
    g_num = 0
    curr_group = [idxs[0]]
    anchor_thumb = thumbs[idxs[0]]

    for k in range(1, len(idxs)):
        i_prev = curr_group[-1]
        i_curr = idxs[k]
        gap = df_clean.loc[i_curr, "ts"] - df_clean.loc[i_prev, "ts"]
        corr_prev, mae_prev = korelasi_dan_mae(thumbs[i_prev], thumbs[i_curr])
        corr_anchor, _ = korelasi_dan_mae(anchor_thumb, thumbs[i_curr])

        # Gabung ke group yang sama hanya jika berdekatan waktu dan mirip secara visual
        if (
            gap <= GROUP_MAX_GAP_SEC
            and corr_prev >= GROUP_MIN_CORR
            and mae_prev <= GROUP_MAX_MAE
            and corr_anchor >= GROUP_MIN_ANCHOR_CORR
        ):
            curr_group.append(i_curr)
        else:
            gid = f"{cls}_G{g_num:04d}"
            for idx_item in curr_group:
                group_ids.append((idx_item, gid))
            if len(curr_group) >= 4 and len(contoh_grup) < 5:
                contoh_grup.append({
                    "group": gid,
                    "kelas": cls,
                    "jumlah": len(curr_group),
                    "contoh_awal": df_clean.loc[curr_group[0], "nama_file"],
                    "contoh_akhir": df_clean.loc[curr_group[-1], "nama_file"],
                })
            g_num += 1
            curr_group = [i_curr]
            anchor_thumb = thumbs[i_curr]

    gid = f"{cls}_G{g_num:04d}"
    for idx_item in curr_group:
        group_ids.append((idx_item, gid))

group_map = dict(group_ids)
df_clean["group"] = df_clean.index.map(group_map)

ringkasan_grup = (
    df_clean.groupby("label")["group"]
    .agg(jumlah_citra="count", jumlah_group="nunique")
    .reindex(CLASSES)
)
ringkasan_grup["rata2_citra_per_group"] = (
    ringkasan_grup["jumlah_citra"] / ringkasan_grup["jumlah_group"]
).round(2)
display(ringkasan_grup)
print("Contoh beberapa group foto beruntun:")
display(pd.DataFrame(contoh_grup))

## 4. Eksplorasi Data (EDA)

In [ ]:
brightness_by_class = {}
for cls in CLASSES:
    vals = []
    for p in df_clean.loc[df_clean["label"] == cls, "abs_path"]:
        gray = cv2.imread(p, cv2.IMREAD_REDUCED_GRAYSCALE_8)
        vals.append(float(gray.mean()))
    brightness_by_class[cls] = vals

counts_clean = df_clean["label"].value_counts().reindex(CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
bars = axes[0].bar(CLASSES, counts_clean.values, color="#3b82f6", edgecolor="black", linewidth=0.6)
axes[0].set_title("Distribusi Jumlah Citra Bersih per Kelas")
axes[0].set_xlabel("Kelas Vokal SIBI")
axes[0].set_ylabel("Jumlah Citra")
axes[0].set_ylim(0, max(counts_clean.values) * 1.16)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)
for bar, c in zip(bars, counts_clean.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 8,
        str(c),
        ha="center",
        va="bottom",
        fontsize=9,
    )

axes[1].boxplot([brightness_by_class[c] for c in CLASSES], tick_labels=CLASSES, patch_artist=True)
axes[1].set_title("Sebaran Kecerahan Citra per Kelas")
axes[1].set_xlabel("Kelas Vokal SIBI")
axes[1].set_ylabel("Rata-Rata Intensitas Piksel (0-255)")
axes[1].set_ylim(0, 255)
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
eda_fig_path = os.path.join(OUTPUT_DIR, "eda_distribusi_dan_kecerahan.png")
plt.savefig(eda_fig_path, dpi=300, bbox_inches="tight")
plt.show()

## 5. Pembagian Data Berbasis Group (64/16/20)

In [ ]:
# Tahap 1 split: pisahkan 20% test dengan StratifiedGroupKFold (5 fold)
sgkf_test = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(
    sgkf_test.split(df_clean, df_clean["label"], df_clean["group"])
)

df_clean["split"] = ""
df_clean.loc[test_idx, "split"] = "test"

# Tahap 2 split: pisahkan sisa 80% menjadi 64% train dan 16% val (1 dari 5 fold)
df_tv = df_clean.loc[trainval_idx].copy()
sgkf_val = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_sub_pos, val_sub_pos = next(
    sgkf_val.split(df_tv, df_tv["label"], df_tv["group"])
)
df_clean.loc[df_tv.index[tr_sub_pos], "split"] = "train"
df_clean.loc[df_tv.index[val_sub_pos], "split"] = "val"

# Validasi integritas split
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    h_a = set(df_clean.loc[df_clean["split"] == a, "hash"])
    h_b = set(df_clean.loc[df_clean["split"] == b, "hash"])
    g_a = set(df_clean.loc[df_clean["split"] == a, "group"])
    g_b = set(df_clean.loc[df_clean["split"] == b, "group"])
    assert len(h_a & h_b) == 0, f"Hash bocor antara {a} dan {b}"
    assert len(g_a & g_b) == 0, f"Group bocor antara {a} dan {b}"

split_order = ["train", "val", "test"]
dist_split = pd.crosstab(df_clean["label"], df_clean["split"]).reindex(index=CLASSES, columns=split_order)
dist_split["total"] = dist_split.sum(axis=1)
dist_split.loc["Total"] = dist_split.sum(axis=0)
assert (dist_split.loc[CLASSES, split_order] > 0).all().all(), "Ada kelas kosong di salah satu split!"

dist_split.to_csv(os.path.join(OUTPUT_DIR, "distribusi_split.csv"))
display(dist_split)

prop = (df_clean["split"].value_counts(normalize=True).reindex(split_order) * 100).round(2)
print("Proporsi aktual (%):", prop.to_dict())

In [ ]:
# Visualisasi distribusi kelas pada data awal dan tiap split
dist_plot_df = pd.DataFrame({
    "Data Awal": df_raw["label"].value_counts().reindex(CLASSES),
    "Train": df_clean.loc[df_clean["split"] == "train", "label"].value_counts().reindex(CLASSES),
    "Validation": df_clean.loc[df_clean["split"] == "val", "label"].value_counts().reindex(CLASSES),
    "Test": df_clean.loc[df_clean["split"] == "test", "label"].value_counts().reindex(CLASSES),
})

fig, ax = plt.subplots(figsize=(10.5, 4.8))
dist_plot_df.plot(kind="bar", ax=ax, edgecolor="black", linewidth=0.5)
ax.set_title("Distribusi Kelas pada Data Awal dan Tiap Split", pad=36)
ax.set_xlabel("Kelas Vokal SIBI")
ax.set_ylabel("Jumlah Citra")
ax.set_xticklabels(CLASSES, rotation=0)
ax.set_ylim(0, dist_plot_df.values.max() * 1.15)
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=4, frameon=True)
for container in ax.containers:
    ax.bar_label(container, fontsize=7.5, padding=2)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "distribusi_kelas_awal_dan_split.png"), dpi=300, bbox_inches="tight")
plt.show()

## 6. Ekstraksi Landmark Tangan (MediaPipe)

In [ ]:
task_path = os.path.join(OLD_EXP_DIR, HAND_TASK_FILENAME)
if not os.path.exists(task_path):
    task_path = os.path.join(OUTPUT_DIR, HAND_TASK_FILENAME)
    if not os.path.exists(task_path):
        print("Mengunduh hand_landmarker.task...")
        urllib.request.urlretrieve(HAND_TASK_URL, task_path)

base_options = mp_python.BaseOptions(model_asset_path=task_path)
mp_options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=MP_NUM_HANDS,
    min_hand_detection_confidence=MP_MIN_DET_CONF,
    min_hand_presence_confidence=MP_MIN_PRES_CONF,
    min_tracking_confidence=MP_MIN_TRACK_CONF,
)
landmarker = mp_vision.HandLandmarker.create_from_options(mp_options)


def extract_hand_landmarks(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result = landmarker.detect(mp_image)
    if not result.hand_landmarks:
        return None
    hand = result.hand_landmarks[0]
    coords = []
    for lm in hand:
        coords.extend([lm.x, lm.y, lm.z])
    return np.array(coords, dtype=np.float32)

In [ ]:
# Ekstraksi landmark langsung dari manifest df_clean (tanpa cache lama)
features_all = []
det_statuses = []

for row in df_clean.itertuples():
    img = cv2.imread(row.abs_path)
    if img is None:
        features_all.append(None)
        det_statuses.append("gagal_baca")
        continue
    feats = extract_hand_landmarks(img)
    if feats is None:
        features_all.append(None)
        det_statuses.append("gagal_deteksi")
    else:
        features_all.append(feats)
        det_statuses.append("terdeteksi")

df_clean["status_deteksi"] = det_statuses
df_clean["feat_idx"] = np.arange(len(df_clean))

# Simpan manifest lengkap (termasuk data yang dikeluarkan pada pemeriksaan awal)
df_keluar = df_raw[df_raw["alasan_dikeluarkan"] != ""].copy()
df_keluar["group"] = ""
df_keluar["split"] = "dikeluarkan"
df_keluar["status_deteksi"] = "tidak_diproses"

cols_manifest = [
    "path_relatif",
    "label",
    "group",
    "hash",
    "split",
    "status_deteksi",
    "alasan_dikeluarkan",
]
df_manifest = pd.concat(
    [df_clean[cols_manifest], df_keluar[cols_manifest]],
    ignore_index=True,
).sort_values(["split", "label", "path_relatif"]).reset_index(drop=True)

manifest_path = os.path.join(OUTPUT_DIR, "manifest_dataset.csv")
df_manifest.to_csv(manifest_path, index=False)
print("Manifest tersimpan di:", manifest_path)

In [ ]:
# Ringkasan keberhasilan dan kegagalan deteksi per kelas dan split
det_rows = []
for sp in split_order:
    for cls in CLASSES:
        sub = df_clean[(df_clean["split"] == sp) & (df_clean["label"] == cls)]
        n_tot = len(sub)
        n_ok = int((sub["status_deteksi"] == "terdeteksi").sum())
        n_fail = n_tot - n_ok
        det_rows.append({
            "split": sp,
            "kelas": cls,
            "total": n_tot,
            "terdeteksi": n_ok,
            "gagal": n_fail,
            "coverage_persen": round(100.0 * n_ok / n_tot, 2) if n_tot else 0.0,
        })

df_det = pd.DataFrame(det_rows)
df_det.to_csv(os.path.join(OUTPUT_DIR, "deteksi_landmark_per_kelas_split.csv"), index=False)
display(df_det)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
cov_pivot = df_det.pivot(index="kelas", columns="split", values="coverage_persen").reindex(index=CLASSES, columns=split_order)
fail_pivot = df_det.pivot(index="kelas", columns="split", values="gagal").reindex(index=CLASSES, columns=split_order)

cov_pivot.plot(kind="bar", ax=axes[0], edgecolor="black", linewidth=0.5)
axes[0].set_title("Cakupan Deteksi Landmark (%) per Kelas dan Split", pad=36)
axes[0].set_xlabel("Kelas Vokal SIBI")
axes[0].set_ylabel("Persentase Terdeteksi (%)")
axes[0].set_xticklabels(CLASSES, rotation=0)
axes[0].set_ylim(90, 102)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)
axes[0].legend(loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=3, frameon=True)

fail_pivot.plot(kind="bar", ax=axes[1], edgecolor="black", linewidth=0.5)
axes[1].set_title("Jumlah Citra Gagal Deteksi per Kelas dan Split", pad=36)
axes[1].set_xlabel("Kelas Vokal SIBI")
axes[1].set_ylabel("Jumlah Citra Gagal")
axes[1].set_xticklabels(CLASSES, rotation=0)
axes[1].set_ylim(0, max(3, fail_pivot.values.max() + 1.5))
axes[1].grid(axis="y", linestyle="--", alpha=0.5)
axes[1].legend(loc="lower center", bbox_to_anchor=(0.5, 1.01), ncol=3, frameon=True)
for container in axes[1].containers:
    axes[1].bar_label(container, fontsize=8, padding=2)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "deteksi_landmark.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Contoh citra dan overlay landmark untuk tiap kelas
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20), (0, 17),
]

fig, axes = plt.subplots(1, 5, figsize=(14, 3.3))
for idx_cls, cls in enumerate(CLASSES):
    row = df_clean[(df_clean["label"] == cls) & (df_clean["status_deteksi"] == "terdeteksi")].iloc[0]
    img_bgr = cv2.imread(row["abs_path"])
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape
    pts = features_all[row["feat_idx"]].reshape(21, 3)

    axes[idx_cls].imshow(img_rgb)
    for a, b in HAND_CONNECTIONS:
        axes[idx_cls].plot(
            [pts[a, 0] * w, pts[b, 0] * w],
            [pts[a, 1] * h, pts[b, 1] * h],
            color="#22c55e",
            linewidth=1.5,
        )
    axes[idx_cls].scatter(pts[:, 0] * w, pts[:, 1] * h, c="#ef4444", s=14, zorder=3)
    axes[idx_cls].set_title(f"Kelas {cls}")
    axes[idx_cls].axis("off")

plt.suptitle("Contoh Citra dan Landmark Tangan per Kelas Vokal SIBI", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "contoh_citra_landmark_per_kelas.png"), dpi=300, bbox_inches="tight")
plt.show()

## 7. Preprocessing dan Augmentasi Landmark

In [ ]:
def ambil_split_terdeteksi(split_nama):
    sub = df_clean[(df_clean["split"] == split_nama) & (df_clean["status_deteksi"] == "terdeteksi")].copy()
    X = np.vstack([features_all[i] for i in sub["feat_idx"]]).astype(np.float32)
    y = sub["label"].values
    return X, y, sub


X_train_raw, y_train_raw, df_train_det = ambil_split_terdeteksi("train")
X_val_raw, y_val_raw, df_val_det = ambil_split_terdeteksi("val")
X_test_raw, y_test_raw, df_test_det = ambil_split_terdeteksi("test")

# Simpan fitur mentah hasil ekstraksi baru di output_final
np.savez_compressed(
    os.path.join(OUTPUT_DIR, "landmarks_extracted.npz"),
    X_train=X_train_raw, y_train=y_train_raw, path_train=df_train_det["path_relatif"].values,
    X_val=X_val_raw, y_val=y_val_raw, path_val=df_val_det["path_relatif"].values,
    X_test=X_test_raw, y_test=y_test_raw, path_test=df_test_det["path_relatif"].values,
)


def normalize_landmarks(X):
    # Fungsi normalisasi yang sama untuk train, validation, test, dan inference
    pts = np.asarray(X, dtype=np.float32).reshape(-1, 21, 3).copy()
    wrist = pts[:, 0:1, :]
    pts = pts - wrist
    xy_dist = np.linalg.norm(pts[:, :, :2], axis=2)
    scale = np.max(xy_dist, axis=1, keepdims=True)
    scale = np.maximum(scale, 1e-6)
    pts[:, :, :2] = pts[:, :, :2] / scale[:, :, None]
    pts[:, :, 2] = pts[:, :, 2] / scale
    return pts.reshape(-1, 63).astype(np.float32)


def augment_landmarks_batch(X, y, n_aug_per_sample=N_AUG_PER_SAMPLE, seed=SEED):
    rng = np.random.default_rng(seed)
    X_aug_list, y_aug_list = [X], [y]

    for _ in range(n_aug_per_sample):
        pts = X.reshape(-1, 21, 3).copy()
        scales = rng.uniform(AUG_SCALE_MIN, AUG_SCALE_MAX, size=(pts.shape[0], 1, 1)).astype(np.float32)
        pts = pts * scales
        noise = rng.normal(0.0, AUG_NOISE_STD, size=pts.shape).astype(np.float32)
        pts = pts + noise
        X_aug_list.append(pts.reshape(-1, 63))
        y_aug_list.append(y.copy())

    X_combined = np.vstack(X_aug_list)
    y_combined = np.concatenate(y_aug_list)
    perm = rng.permutation(len(y_combined))
    return X_combined[perm], y_combined[perm]


le = LabelEncoder()
le.fit(CLASSES)
num_classes = len(CLASSES)

y_train_enc = le.transform(y_train_raw)
y_val_enc = le.transform(y_val_raw)
y_test_enc = le.transform(y_test_raw)

X_train_norm = normalize_landmarks(X_train_raw)
X_val_norm = normalize_landmarks(X_val_raw)
X_test_norm = normalize_landmarks(X_test_raw)

y_train_cat = keras.utils.to_categorical(y_train_enc, num_classes)
y_val_cat = keras.utils.to_categorical(y_val_enc, num_classes)
y_test_cat = keras.utils.to_categorical(y_test_enc, num_classes)

# Hitung class weight hanya dari data train
cw_vals = compute_class_weight(
    class_weight="balanced", classes=np.arange(num_classes), y=y_train_enc
)
class_weights_train = {int(k): float(v) for k, v in enumerate(cw_vals)}

# Augmentasi hanya diterapkan pada train (untuk E1)
X_train_aug, y_train_aug_enc = augment_landmarks_batch(X_train_norm, y_train_enc)
y_train_aug_cat = keras.utils.to_categorical(y_train_aug_enc, num_classes)

print(f"Train terdeteksi : {X_train_norm.shape} -> setelah augmentasi: {X_train_aug.shape}")
print(f"Val terdeteksi   : {X_val_norm.shape}")
print(f"Test terdeteksi  : {X_test_norm.shape}")
print("Class weights    :", {CLASSES[k]: round(v, 4) for k, v in class_weights_train.items()})

In [ ]:
# Visualisasi landmark sebelum dan sesudah normalisasi serta augmentasi
sample_raw = X_train_raw[0:1]
sample_norm = normalize_landmarks(sample_raw)
rng_vis = np.random.default_rng(SEED)
pts_aug = sample_norm.reshape(1, 21, 3).copy()
pts_aug = pts_aug * rng_vis.uniform(AUG_SCALE_MIN, AUG_SCALE_MAX, size=(1, 1, 1)).astype(np.float32)
pts_aug = pts_aug + rng_vis.normal(0.0, AUG_NOISE_STD, size=pts_aug.shape).astype(np.float32)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.0))
panels = [
    (sample_raw.reshape(21, 3), f"Sebelum Normalisasi (Kelas {y_train_raw[0]})"),
    (sample_norm.reshape(21, 3), "Sesudah Normalisasi (Pusat Wrist)"),
    (pts_aug.reshape(21, 3), "Sesudah Augmentasi (Skala + Jitter)"),
]

for ax, (pts, title) in zip(axes, panels):
    for a, b in HAND_CONNECTIONS:
        ax.plot([pts[a, 0], pts[b, 0]], [pts[a, 1], pts[b, 1]], color="#3b82f6", linewidth=1.4)
    ax.scatter(pts[:, 0], pts[:, 1], c="#ef4444", s=18, zorder=3)
    ax.scatter([pts[0, 0]], [pts[0, 1]], c="#16a34a", s=45, zorder=4, label="Wrist (0)")
    ax.invert_yaxis()
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Koordinat X")
    ax.set_ylabel("Koordinat Y")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(loc="best", fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "landmark_normalisasi_dan_augmentasi.png"), dpi=300, bbox_inches="tight")
plt.show()

## 8. Training Model E0 (Baseline)

In [ ]:
set_all_seeds(SEED)

model_e0 = keras.Sequential([
    layers.Input(shape=(63,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation="softmax"),
], name="MLP_E0_Baseline")

model_e0.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_e0 = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=EARLY_STOP_PATIENCE, restore_best_weights=True
    )
]

history_e0 = model_e0.fit(
    X_train_norm,
    y_train_cat,
    validation_data=(X_val_norm, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights_train,
    callbacks=callbacks_e0,
    verbose=1,
)

val_loss_e0, val_acc_e0 = model_e0.evaluate(X_val_norm, y_val_cat, verbose=0)
val_pred_e0 = np.argmax(model_e0.predict(X_val_norm, verbose=0), axis=1)
val_f1_e0 = f1_score(y_val_enc, val_pred_e0, average="macro")
print(f"[E0 Validation] Loss: {val_loss_e0:.4f} | Accuracy: {val_acc_e0:.4f} | Macro-F1: {val_f1_e0:.4f}")

## 9. Training Model E1 (Augmentasi Landmark + Batch Normalization)

Model E1 mengubah augmentasi data train dan menambahkan Batch Normalization secara bersamaan, sehingga pengaruh masing-masing komponen tidak dipisahkan.

In [ ]:
set_all_seeds(SEED)

model_e1 = keras.Sequential([
    layers.Input(shape=(63,)),
    layers.Dense(128, activation=None),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation=None),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation="softmax"),
], name="MLP_E1_Aug_BN")

model_e1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_e1 = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=EARLY_STOP_PATIENCE, restore_best_weights=True
    )
]

history_e1 = model_e1.fit(
    X_train_aug,
    y_train_aug_cat,
    validation_data=(X_val_norm, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights_train,
    callbacks=callbacks_e1,
    verbose=1,
)

val_loss_e1, val_acc_e1 = model_e1.evaluate(X_val_norm, y_val_cat, verbose=0)
val_pred_e1 = np.argmax(model_e1.predict(X_val_norm, verbose=0), axis=1)
val_f1_e1 = f1_score(y_val_enc, val_pred_e1, average="macro")
print(f"[E1 Validation] Loss: {val_loss_e1:.4f} | Accuracy: {val_acc_e1:.4f} | Macro-F1: {val_f1_e1:.4f}")

In [ ]:
# Plot kurva loss dan accuracy E0 serta E1 dengan skala sumbu yang sebanding
max_loss = max(
    max(history_e0.history["loss"]),
    max(history_e0.history["val_loss"]),
    max(history_e1.history["loss"]),
    max(history_e1.history["val_loss"]),
)
min_acc = min(
    min(history_e0.history["accuracy"]),
    min(history_e0.history["val_accuracy"]),
    min(history_e1.history["accuracy"]),
    min(history_e1.history["val_accuracy"]),
)

def plot_kurva_training(hist, nama_model, filename):
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
    epochs_range = range(1, len(hist.history["loss"]) + 1)

    axes[0].plot(epochs_range, hist.history["loss"], label="Train Loss")
    axes[0].plot(epochs_range, hist.history["val_loss"], label="Validation Loss")
    axes[0].set_title(f"Kurva Loss Train dan Validation ({nama_model})")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_ylim(0, max_loss * 1.05)
    axes[0].grid(True, linestyle="--", alpha=0.5)
    axes[0].legend()

    axes[1].plot(epochs_range, hist.history["accuracy"], label="Train Accuracy")
    axes[1].plot(epochs_range, hist.history["val_accuracy"], label="Validation Accuracy")
    axes[1].set_title(f"Kurva Akurasi Train dan Validation ({nama_model})")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Akurasi")
    axes[1].set_ylim(max(0.0, min_acc - 0.05), 1.02)
    axes[1].grid(True, linestyle="--", alpha=0.5)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300, bbox_inches="tight")
    plt.show()


plot_kurva_training(history_e0, "E0 Baseline", "kurva_training_e0.png")
plot_kurva_training(history_e1, "E1 Aug + BN", "kurva_training_e1.png")

## 10. Pemilihan Model Berdasarkan Validation Set

In [ ]:
df_val_comp = pd.DataFrame([
    {
        "model": "E0",
        "val_macro_f1": val_f1_e0,
        "val_accuracy": val_acc_e0,
        "val_loss": val_loss_e0,
        "epoch_terbaik": int(np.argmin(history_e0.history["val_loss"]) + 1),
    },
    {
        "model": "E1",
        "val_macro_f1": val_f1_e1,
        "val_accuracy": val_acc_e1,
        "val_loss": val_loss_e1,
        "epoch_terbaik": int(np.argmin(history_e1.history["val_loss"]) + 1),
    },
])
df_val_comp.to_csv(os.path.join(OUTPUT_DIR, "perbandingan_validasi.csv"), index=False)
display(df_val_comp)

# Aturan pemilihan: macro-F1 validation tertinggi, seri -> validation loss terendah
if val_f1_e1 > val_f1_e0 or (np.isclose(val_f1_e1, val_f1_e0) and val_loss_e1 < val_loss_e0):
    selected_model_name = "E1"
    selected_model = model_e1
else:
    selected_model_name = "E0"
    selected_model = model_e0

print(f"Model terpilih sebelum evaluasi test: {selected_model_name}")

## 11. Evaluasi Final pada Test Set

Catatan: dataset ini pernah digunakan pada eksperimen sebelumnya sehingga test set baru tidak sepenuhnya belum pernah dilihat secara historis.

In [ ]:
n_test_total = int((df_clean["split"] == "test").sum())
n_test_det = len(y_test_enc)
n_test_fail = n_test_total - n_test_det
test_coverage = n_test_det / n_test_total


def evaluasi_model_test(model, nama):
    loss_det, acc_det = model.evaluate(X_test_norm, y_test_cat, verbose=0)
    probs = model.predict(X_test_norm, verbose=0)
    pred = np.argmax(probs, axis=1)
    macro_f1 = f1_score(y_test_enc, pred, average="macro")
    n_correct = int(np.sum(pred == y_test_enc))
    n_wrong_det = int(np.sum(pred != y_test_enc))
    acc_e2e = n_correct / n_test_total
    return {
        "model": nama,
        "n_test_total": n_test_total,
        "n_terdeteksi": n_test_det,
        "n_gagal_deteksi": n_test_fail,
        "coverage": test_coverage,
        "akurasi_terdeteksi": acc_det,
        "macro_f1_terdeteksi": macro_f1,
        "loss_terdeteksi": loss_det,
        "salah_klasifikasi": n_wrong_det,
        "benar_total": n_correct,
        "akurasi_end_to_end": acc_e2e,
    }, probs, pred


res_e0, probs_test_e0, pred_test_e0 = evaluasi_model_test(model_e0, "E0")
res_e1, probs_test_e1, pred_test_e1 = evaluasi_model_test(model_e1, "E1")

df_test_comp = pd.DataFrame([res_e0, res_e1])
df_test_comp.to_csv(os.path.join(OUTPUT_DIR, "perbandingan_test_e0_e1.csv"), index=False)
display(df_test_comp)

print("\nClassification Report E0 (sampel terdeteksi):")
print(classification_report(y_test_enc, pred_test_e0, target_names=CLASSES, digits=4))
print("Classification Report E1 (sampel terdeteksi):")
print(classification_report(y_test_enc, pred_test_e1, target_names=CLASSES, digits=4))

In [ ]:
# Confusion matrix E0 dan E1 dengan skala warna seragam
cm_e0 = confusion_matrix(y_test_enc, pred_test_e0)
cm_e1 = confusion_matrix(y_test_enc, pred_test_e1)
vmax_cm = max(cm_e0.max(), cm_e1.max())

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
for ax, cm, judul in [
    (axes[0], cm_e0, "Confusion Matrix Test - E0 Baseline"),
    (axes[1], cm_e1, "Confusion Matrix Test - E1 Aug + BN"),
]:
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        vmin=0,
        vmax=vmax_cm,
        xticklabels=CLASSES,
        yticklabels=CLASSES,
        ax=ax,
    )
    ax.set_title(judul)
    ax.set_xlabel("Label Prediksi")
    ax.set_ylabel("Label Sebenarnya")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix_test_e0_e1.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Tabel dan grafik precision, recall, F1 per kelas (E0 vs E1)
rows_cls = []
for nama, pred in [("E0", pred_test_e0), ("E1", pred_test_e1)]:
    p, r, f1, sup = precision_recall_fscore_support(y_test_enc, pred, labels=np.arange(num_classes))
    for i, cls in enumerate(CLASSES):
        rows_cls.append({
            "model": nama,
            "kelas": cls,
            "precision": float(p[i]),
            "recall": float(r[i]),
            "f1_score": float(f1[i]),
            "support": int(sup[i]),
        })

df_cls_metrics = pd.DataFrame(rows_cls)
df_cls_metrics.to_csv(os.path.join(OUTPUT_DIR, "metrik_per_kelas_test.csv"), index=False)
display(df_cls_metrics)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2), sharey=True)
for ax, metrik, label_m in [
    (axes[0], "precision", "Precision"),
    (axes[1], "recall", "Recall"),
    (axes[2], "f1_score", "F1-Score"),
]:
    piv = df_cls_metrics.pivot(index="kelas", columns="model", values=metrik).reindex(CLASSES)
    piv.plot(kind="bar", ax=ax, edgecolor="black", linewidth=0.5)
    ax.set_title(f"{label_m} per Kelas (Test Set)")
    ax.set_xlabel("Kelas Vokal SIBI")
    ax.set_ylabel(label_m)
    ax.set_xticklabels(CLASSES, rotation=0)
    ax.set_ylim(0, 1.08)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.legend(title="Model", loc="lower right")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "metrik_per_kelas_test.png"), dpi=300, bbox_inches="tight")
plt.show()

## 12. Analisis Kesalahan dan Kegagalan Deteksi

In [ ]:
err_rows = []
for i in range(len(y_test_enc)):
    true_cls = CLASSES[y_test_enc[i]]
    p_e0_cls = CLASSES[pred_test_e0[i]]
    p_e1_cls = CLASSES[pred_test_e1[i]]
    if p_e0_cls != true_cls or p_e1_cls != true_cls:
        err_rows.append({
            "jenis": "salah_klasifikasi",
            "path_relatif": df_test_det.iloc[i]["path_relatif"],
            "abs_path": df_test_det.iloc[i]["abs_path"],
            "label_asli": true_cls,
            "pred_e0": p_e0_cls,
            "conf_e0": round(float(probs_test_e0[i, pred_test_e0[i]]), 4),
            "pred_e1": p_e1_cls,
            "conf_e1": round(float(probs_test_e1[i, pred_test_e1[i]]), 4),
        })

df_test_fail = df_clean[(df_clean["split"] == "test") & (df_clean["status_deteksi"] != "terdeteksi")]
for row in df_test_fail.itertuples():
    err_rows.append({
        "jenis": row.status_deteksi,
        "path_relatif": row.path_relatif,
        "abs_path": row.abs_path,
        "label_asli": row.label,
        "pred_e0": "-",
        "conf_e0": np.nan,
        "pred_e1": "-",
        "conf_e1": np.nan,
    })

df_errors = pd.DataFrame(err_rows)
df_errors.drop(columns=["abs_path"], errors="ignore").to_csv(
    os.path.join(OUTPUT_DIR, "daftar_kesalahan_test.csv"), index=False
)
print(f"Total kasus kesalahan/gagal deteksi di test: {len(df_errors)}")
if len(df_errors) > 0:
    display(df_errors.drop(columns=["abs_path"]))

    n_show = min(len(df_errors), 8)
    ncols = min(4, n_show)
    nrows = int(np.ceil(n_show / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 3.6 * nrows))
    axes_flat = np.atleast_1d(axes).ravel()
    for ax in axes_flat[n_show:]:
        ax.axis("off")

    for k in range(n_show):
        r = df_errors.iloc[k]
        img_rgb = cv2.cvtColor(cv2.imread(r["abs_path"]), cv2.COLOR_BGR2RGB)
        axes_flat[k].imshow(img_rgb)
        if r["jenis"] == "salah_klasifikasi":
            t = f"Asli: {r['label_asli']}\nE0: {r['pred_e0']} ({r['conf_e0']:.2f}) | E1: {r['pred_e1']} ({r['conf_e1']:.2f})"
        else:
            t = f"Asli: {r['label_asli']}\nStatus: {r['jenis']}"
        axes_flat[k].set_title(t, fontsize=8.5)
        axes_flat[k].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "analisis_kesalahan_test.png"), dpi=300, bbox_inches="tight")
    plt.show()

## 13. Penyimpanan Model, Verifikasi Reload, dan Contoh Inference

In [ ]:
path_e0 = os.path.join(OUTPUT_DIR, "model_e0.keras")
path_e1 = os.path.join(OUTPUT_DIR, "model_e1.keras")
path_sel = os.path.join(OUTPUT_DIR, "model_terpilih.keras")

model_e0.save(path_e0)
model_e1.save(path_e1)
selected_model.save(path_sel)

label_map = {int(i): str(cls) for i, cls in enumerate(le.classes_)}
with open(os.path.join(OUTPUT_DIR, "label_map.json"), "w", encoding="utf-8") as f:
    json.dump(label_map, f, indent=2)

preproc_config = {
    "model_terpilih": selected_model_name,
    "num_features": 63,
    "num_landmarks": 21,
    "normalization": "translasi_ke_wrist_0_dan_skala_jarak_xy_maksimum",
    "scale_eps": 1e-6,
    "mediapipe": {
        "num_hands": MP_NUM_HANDS,
        "min_hand_detection_confidence": MP_MIN_DET_CONF,
        "min_hand_presence_confidence": MP_MIN_PRES_CONF,
        "min_tracking_confidence": MP_MIN_TRACK_CONF,
    },
}
with open(os.path.join(OUTPUT_DIR, "preprocessing_config.json"), "w", encoding="utf-8") as f:
    json.dump(preproc_config, f, indent=2)

history_data = {
    "E0": {k: [float(v) for v in vals] for k, vals in history_e0.history.items()},
    "E1": {k: [float(v) for v in vals] for k, vals in history_e1.history.items()},
}
with open(os.path.join(OUTPUT_DIR, "training_history.json"), "w", encoding="utf-8") as f:
    json.dump(history_data, f, indent=2)

# Muat ulang model terpilih dan pastikan prediksi sama persis
reloaded_model = keras.models.load_model(path_sel)
probs_before = selected_model.predict(X_test_norm, verbose=0)
probs_after = reloaded_model.predict(X_test_norm, verbose=0)
max_diff = float(np.max(np.abs(probs_before - probs_after)))
assert np.allclose(probs_before, probs_after, atol=1e-6), "Prediksi setelah reload berbeda!"
assert np.array_equal(np.argmax(probs_before, axis=1), np.argmax(probs_after, axis=1))
print(f"Verifikasi reload model terpilih ({selected_model_name}) berhasil. Max diff probabilitas: {max_diff:.2e}")

In [ ]:
def predict_single_image(image_path, model, mp_landmarker, lbl_map):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        return {"status": "gagal_baca", "prediksi": None, "confidence": 0.0}

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result = mp_landmarker.detect(mp_image)
    if not result.hand_landmarks:
        return {"status": "gagal_deteksi_tangan", "prediksi": None, "confidence": 0.0}

    coords = []
    for lm in result.hand_landmarks[0]:
        coords.extend([lm.x, lm.y, lm.z])
    x_raw = np.array(coords, dtype=np.float32).reshape(1, 63)
    x_norm = normalize_landmarks(x_raw)
    probs = model.predict(x_norm, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    return {
        "status": "berhasil",
        "prediksi": lbl_map[pred_idx],
        "confidence": round(float(probs[pred_idx]), 4),
    }


# Contoh 1: citra test yang terdeteksi
contoh_ok = df_test_det.iloc[0]
hasil_ok = predict_single_image(contoh_ok["abs_path"], reloaded_model, landmarker, label_map)
print("Inference contoh terdeteksi :", contoh_ok["path_relatif"], "| Asli:", contoh_ok["label"], "->", hasil_ok)

# Contoh 2: citra tanpa tangan terdeteksi (dari test gagal deteksi jika ada, atau citra kosong)
if len(df_test_fail) > 0:
    contoh_fail = df_test_fail.iloc[0]
    hasil_fail = predict_single_image(contoh_fail["abs_path"], reloaded_model, landmarker, label_map)
    print("Inference contoh tanpa tangan:", contoh_fail["path_relatif"], "| Asli:", contoh_fail["label"], "->", hasil_fail)
else:
    blank_path = os.path.join(OUTPUT_DIR, "_contoh_tanpa_tangan.png")
    cv2.imwrite(blank_path, np.full((240, 240, 3), 255, dtype=np.uint8))
    hasil_fail = predict_single_image(blank_path, reloaded_model, landmarker, label_map)
    print("Inference contoh citra kosong:", hasil_fail)

In [ ]:
# Buat requirements.txt berisi library yang benar-benar dipakai
pkg_names = [
    "numpy",
    "pandas",
    "opencv-python",
    "pillow",
    "matplotlib",
    "seaborn",
    "mediapipe",
    "scikit-learn",
    "tensorflow",
]
req_lines = [f"{pkg}=={importlib.metadata.version(pkg)}" for pkg in pkg_names]
req_path = os.path.join(BASE_DIR, "requirements.txt")
with open(req_path, "w", encoding="utf-8") as f:
    f.write("\n".join(req_lines) + "\n")

print("requirements.txt tersimpan di:", req_path)
print("\n".join(req_lines))